## Feature Score

Goal: Combining all feature selections from the different models as well as their scores. 
1. load all df
2. normalize each
3. add the scores
4. normalize results
   

In [3]:
#import needed packages
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [4]:
#Read in Data and check 
PM = pd.read_csv("data/featureselection_PM_score_2012.csv")
CV = pd.read_csv("data/featureselection_cramersV_LGBMscore_2012.csv")
gini = pd.read_csv("data/featureselection_gini_score_2012.csv")

print(f" permutation: \n {PM.head()}")
print(f" gini: \n {gini.head()}")
print(f" Cramers V: \n {CV.head()}")

 permutation: 
    Unnamed: 0              base_name  imp_grouped
0          23  vote_generic_baseline     0.004077
1           1   cmatch_romn_baseline     0.003851
2           0   cmatch_ging_baseline     0.002945
3           7     fav_obama_baseline     0.002492
4          12      obamaapp_baseline     0.001812
 gini: 
    Unnamed: 0              base_name  imp_grouped
0          12   cmatch_ging_baseline     0.048013
1          13   cmatch_romn_baseline     0.043604
2         193    presvote08_baseline     0.041226
3         190   post_presvote12_2012     0.039811
4         260  vote_generic_baseline     0.036162
 Cramers V: 
    Unnamed: 0  n_features              added_feature  auc_lr_train  \
0           0           1       post_presvote12_2012        0.9239   
1           1           2       cmatch_romn_baseline        0.9370   
2           2           3      vote_generic_baseline        0.9390   
3           3           4          obamaapp_baseline        0.9443   
4          

In [5]:
# normalize needed columns with MinMaxScaler()
# source: https://www.geeksforgeeks.org/python/normalize-a-column-in-pandas/

PM["norm_score_PM"] = MinMaxScaler().fit_transform(np.array(PM["imp_grouped"]).reshape(-1,1))
gini["norm_score_gini"] = MinMaxScaler().fit_transform(np.array(gini["imp_grouped"]).reshape(-1,1))
CV["norm_score_CV"] = MinMaxScaler().fit_transform(np.array(CV["auc_lr_test"]).reshape(-1,1)) #could also use: auc_lgbm_test



In [6]:
#check df
print(f" permutation: \n {PM.head()}")
print(f" gini: \n {gini.head()}")
print(f" Cramers V: \n {CV.head()}")

 permutation: 
    Unnamed: 0              base_name  imp_grouped  norm_score_PM
0          23  vote_generic_baseline     0.004077       1.000000
1           1   cmatch_romn_baseline     0.003851       0.944444
2           0   cmatch_ging_baseline     0.002945       0.722222
3           7     fav_obama_baseline     0.002492       0.611111
4          12      obamaapp_baseline     0.001812       0.444444
 gini: 
    Unnamed: 0              base_name  imp_grouped  norm_score_gini
0          12   cmatch_ging_baseline     0.048013         1.000000
1          13   cmatch_romn_baseline     0.043604         0.908170
2         193    presvote08_baseline     0.041226         0.858653
3         190   post_presvote12_2012     0.039811         0.829163
4         260  vote_generic_baseline     0.036162         0.753182
 Cramers V: 
    Unnamed: 0  n_features              added_feature  auc_lr_train  \
0           0           1       post_presvote12_2012        0.9239   
1           1           2    

In [7]:
#rename feature names to match, so .merge works 
PM_ = PM.rename(columns={"base_name": "feature"})
gini_ = gini.rename(columns={"base_name": "feature"})
CV_ = CV.rename(columns={"added_feature": "feature"})

In [35]:
#merge df and add scores together
# with the help of ChatGPT
# Prompts: 
"""
#PM["combined_scores"] = CV["norm_score"] + PM["norm_score"] + gini["norm_score"]
This does not work as the columns don't match by index. So I asked ChatGPT to fix it using this: 

I want to add some scores of PDS: 
PM["combined_scores"] = CV["norm_score"] + PM["norm_score"] + gini["norm_score"] //
BUT not all my rows match, and there are some rows that exist in one df but not in the other. 
Can you fix it and create a new df named FSC (feature selection score

i have 3 df with similar features.
the features are ordered differently and have slightly different names. 
I want to create a new df that combines the values for each feature into a new df 
-> i.e. df one as Apple: 1, Pear: 0.5, Kiwi: 0.4 CV has Pear: 0.7, Cherry: 1, Apple: 0.4, Orange: 0.8 
and it should return: combined_score which should look like this: 
Apple: 1.4, Kiwi: 0.4, Pear: 1.3, Cherry: 1, Orange: 0.8
"""

In [8]:
#merge df and add scores together
# with the help of ChatGPT
"""
#cannot combine df on feature name when the following code is applied: 
PM_ = PM[["feature", "norm_score_PM"]]
gini_ = gini[["feature", "norm_score_gini"]]
CV_ = CV[["feature", "norm_score_CV"]]
"""
# --- Step 3: Merge all features (outer join!) ---
FSC = PM_.merge(gini_, on="feature", how="outer")
FSC = FSC.merge(CV_, on="feature", how="outer")

# --- Step 4: Replace missing values with 0 ---
FSC = FSC.fillna(0)

# --- Step 5: Create combined score ---
FSC["combined_score"] = (
    FSC["norm_score_PM"] +
    FSC["norm_score_gini"] +
    FSC["norm_score_CV"]
)

# --- Step 6: Sort features by importance ---
FSC = FSC.sort_values("combined_score").reset_index(drop=True)

# Optional: keep only relevant columns
FSC = FSC[["feature", "combined_score"]]
print(FSC)


                        feature  combined_score
0     volunteerorg2_10_baseline        0.000000
1                         therm        0.000000
2           healthdk_0_baseline        0.000000
3             student2_baseline        0.000000
4    org_membership_15_baseline        0.000000
..                          ...             ...
261          taxwealth_baseline        1.451524
262   healthreformbill_baseline        1.523873
263        cmatch_ging_baseline        1.722222
264       vote_generic_baseline        1.944561
265        cmatch_romn_baseline        2.123304

[266 rows x 2 columns]


In [9]:
# normalize and save final feature selection and scores
FSC["norm_score_c"] = MinMaxScaler().fit_transform(np.array(FSC["combined_score"]).reshape(-1,1))
FSC.to_csv("data/featureselection_combined_scores_2012.csv")

## further filerting: 
1. only keep top 20
2. remove variables that did not improve the test accuracy as viewd in the plots 

In [10]:
# filter for top 20
df = FSC.iloc[245:266]
print(df)
len(df)
#UNSURE ABOUT VARIABLES: 
#ombaapp -> do you like how obama is running the country
#post_bid3 -> do you see yourself as democrat/ republican/ independent...

#envser2_baseline -> removed, too many NA


                       feature  combined_score  norm_score_c
245        teapartsup_baseline        1.091868      0.514231
246          fav_sant_baseline        1.127245      0.530892
247             track_baseline        1.166465      0.549363
248         fav_biden_baseline        1.175067      0.553414
249          likenewt_baseline        1.175409      0.553575
250         fav_obama_baseline        1.196619      0.563565
251      healthtaxch3_baseline        1.207399      0.568641
252          obamaapp_baseline        1.214359      0.571919
253             post_pid3_2012        1.235466      0.581860
254         saysobama_baseline        1.248060      0.587791
255          fav_perr_baseline        1.262867      0.594765
256          fav_ging_baseline        1.286894      0.606081
257           envser2_baseline        1.287060      0.606159
258           envwarm_baseline        1.312760      0.618263
259          fav_bach_baseline        1.316318      0.619939
260          post_house1

21

In [27]:
#go through questtionair meta data and decide what features to drop 
ft_toremove = ["fav_sant_baseline","fav_biden_baseline", "likenewt_baseline", "fav_obama_baseline","saysobama_baseline", "fav_perr_baseline", "fav_ging_baseline", "fav_bach_baseline", "post_house12_2012", "cmatch_ging_baseline", "vote_generic_baseline","envser2_baseline", "cmatch_romn_baseline"]

In [28]:
df = df[~df["feature"].isin(ft_toremove)]
print(df)
#print(len(df)) #9
df.to_csv("data/featureselection_combined_scores_filtered_2012.csv")

                       feature  combined_score  norm_score_c
245        teapartsup_baseline        1.091868      0.514231
247             track_baseline        1.166465      0.549363
251      healthtaxch3_baseline        1.207399      0.568641
252          obamaapp_baseline        1.214359      0.571919
253             post_pid3_2012        1.235466      0.581860
258           envwarm_baseline        1.312760      0.618263
261         taxwealth_baseline        1.451524      0.683615
262  healthreformbill_baseline        1.523873      0.717689


In [29]:
#make df with selected variables
fts = df['feature'].values.tolist() + ["presvote16post_2016"]
#print(fts)

df_total = pd.read_csv("data/VOTER_Survey_December16_Release1.csv")

df_selectedfeatures_2012 = df_total[fts]
df_selectedfeatures_2012.to_csv("data/selectedfeatures_2012_total.csv") # no Na :)

/tmp/ipykernel_214/1981068587.py:5: DtypeWarning: Columns (0: votereg_fnd_baseline, 1: religpew_muslim_baseline) have mixed types. Specify dtype option on import or set low_memory=False.
  df_total = pd.read_csv("data/VOTER_Survey_December16_Release1.csv")
